In [ ]:
import os
import json
import shutil
import torch
import torchvision
from torch.utils.data import DataLoader
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.datasets import CocoDetection
from torchvision.transforms import functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.tensorboard import SummaryWriter
from torchvision.ops import box_iou
from pathlib import Path
from torchvision.transforms import functional as F
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from pytorch_lightning.callbacks import EarlyStopping

In [16]:
!pip install pycocotools


In [ ]:
json_path = "annotations.json"
output_root = ""

# Criação de pastas
splits = ['train', 'val', 'test']
for split in splits:
    os.makedirs(os.path.join(output_root, "images", split), exist_ok=True)

# Carregar ficheiro JSON
with open(json_path, 'r') as f:
    data = json.load(f)

# Criar índice por id para acesso rápido
id_to_image = {img['id']: img for img in data['images']}

annotations = data['annotations']['pieces']

# Criação de índice de anotações por imagem_id
annotations_by_image = {}
for ann in annotations:
    image_id = ann['image_id']
    if ann.get('bbox'):
        bbox = ann['bbox']
    else:
        continue
    annotations_by_image.setdefault(image_id, []).append(bbox)

# Estrutura base do COCO
coco_base = {
    "info": {
        "year": "2025",
        "version": "1",
        "description": "Chess Pieces Detect",
        "contributor": "CM,GB,DM,MB",
        "url": "",
        "date_created": "2025-06-11T14:30:00+00:00"
    },
    "licenses": [
        {
            "id": 1,
            "url": "https://creativecommons.org/publicdomain/zero/1.0/",
            "name": "Public Domain"
        }
    ],
    "categories": [
        {"id": 1, "name": "Chess Piece", "supercategory": "none"},
    ]
}

# Processar splits
for split in splits:
    coco_output = coco_base.copy()
    coco_output["images"] = []
    coco_output["annotations"] = []
    annotation_id = 0

    ids = data['splits']['chessred2k'].get(split, [])
    for image_id in ids['image_ids']:
        image_info = id_to_image.get(image_id)
        if not image_info:
            print(f"[AVISO] ID {image_id} não encontrado em 'images'")
            continue

        src_path = "chessred2k/" + image_info.get('path')
        if not src_path or not os.path.isfile(src_path):
            print(f"[AVISO] Ficheiro não encontrado: {src_path}")
            continue

        dst_path = os.path.join(output_root, "images", split, image_info['file_name'])
        shutil.copy2(src_path, dst_path)

        coco_output["images"].append({
            "id": image_id,
            "license": 1,
            "file_name": image_info['file_name'],
            "height": image_info['height'],
            "width": image_info['width'],
        })

        for ann in annotations_by_image.get(image_id, []):
            x, y, w, h = ann
            coco_output["annotations"].append({
                "id": annotation_id,
                "image_id": image_id,
                "category_id": 0,
                "bbox": [x, y, w, h],
                "area": w * h,
                "segmentation": [],
                "iscrowd": 0
            })
            annotation_id += 1

    # Guardar ficheiro JSON por split
    ann_dir = os.path.join(output_root, "images", split)
    os.makedirs(ann_dir, exist_ok=True)
    with open(os.path.join(ann_dir, f"annotations.json"), 'w') as out_json:
        json.dump(coco_output, out_json, indent=4)

In [12]:
# Define transformations
class CocoTransform:
    def __call__(self, image, target):
        image = F.to_tensor(image)  # Convert PIL image to tensor
        return image, target

In [13]:


# --------------- Dataset ----------------
class DummyChessDataset:
    def __init__(self, root="images/"):
        self.image_paths = list(Path(root).glob("*.jpg"))
        self.boxes = [torch.tensor([[50, 60, 120, 160]], dtype=torch.float32) for _ in self.image_paths]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        img_tensor = F.to_tensor(img)
        return img_tensor, self.boxes[idx]

# --------------- Metrics ----------------
def evaluate_map(pred_boxes, pred_scores, gt_boxes, iou_thresh=0.5):
    TP, FP, FN = 0, 0, 0
    for preds, scores, targets in zip(pred_boxes, pred_scores, gt_boxes):
        preds = preds[scores > 0.5]
        matched = []
        for t in targets:
            best_iou = 0
            for i, p in enumerate(preds):
                if i in matched: continue
                iou = box_iou(p.unsqueeze(0), t.unsqueeze(0)).item()
                if iou > best_iou:
                    best_iou = iou
                    best_idx = i
            if best_iou >= iou_thresh:
                TP += 1
                matched.append(best_idx)
            else:
                FN += 1
        FP += len(preds) - len(matched)
    precision = TP / (TP + FP + 1e-6)
    recall = TP / (TP + FN + 1e-6)
    return precision, recall, TP, FP, FN

# --------------- Inference ----------------
def run_frcnn_eval():
    model = fasterrcnn_resnet50_fpn(pretrained=True).eval()
    dataset = DummyChessDataset()

    preds_all, scores_all, targets_all = [], [], []

    for i in range(len(dataset)):
        img, gt_box = dataset[i]
        with torch.no_grad():
            output = model([img])[0]
        boxes = output['boxes'].cpu()
        scores = output['scores'].cpu()

        preds_all.append(boxes)
        scores_all.append(scores)
        targets_all.append(gt_box)

    p, r, TP, FP, FN = evaluate_map(preds_all, scores_all, targets_all)
    print(f"Faster R-CNN → P: {p:.3f}, R: {r:.3f}, TP: {TP}, FP: {FP}, FN: {FN}")



In [17]:
def get_coco_dataset(img_dir, ann_file):
    return CocoDetection(
        root=img_dir,
        annFile=ann_file,
        transforms=CocoTransform()
    )

# Load datasets
train_dataset = get_coco_dataset(
    img_dir="images/train",
    ann_file="images/train/annotations.json"
)


val_dataset = get_coco_dataset(
    img_dir="images/val",
    ann_file="images/val/annotations.json"
)

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

loading annotations into memory...
Done (t=0.28s)
creating index...
index created!
loading annotations into memory...
Done (t=0.09s)
creating index...
index created!


In [18]:
# Load Faster R-CNN with ResNet-50 backbone
def get_model(num_classes):
    # Load pre-trained Faster R-CNN
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
    
    # Get the number of input features for the classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    
    # Replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

In [19]:
num_classes = 1
model = get_model(num_classes)

c:\Users\Dell\OneDrive - Confidencial Imobiliario\Documentos\ComputerVision\Task 3\1st_part\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Dell\OneDrive - Confidencial Imobiliario\Documentos\ComputerVision\Task 3\1st_part\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to C:\Users\Dell/.cache\torch\hub\checkpoints\fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:04<00:00, 36.0MB/s] 


In [20]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

# Define optimizer and learning rate scheduler
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

In [ ]:
def train_one_epoch(model, optimizer, data_loader, device, epoch, writer=None):
    model.train()
    total_loss = 0.0
    count = 0  # batches com targets válidos

    for step, (images, targets) in enumerate(data_loader):
        # Move images para o dispositivo
        images = [img.to(device) for img in images]

        # Processar os targets
        processed_targets = []
        valid_images = []

        for i, target in enumerate(targets):
            boxes = []
            labels = []
            for obj in target:
                bbox = obj.get("bbox", [])
                if len(bbox) != 4:
                    continue
                x, y, w, h = bbox
                if w > 0 and h > 0:
                    boxes.append([x, y, x + w, y + h])
                    labels.append(obj["category_id"])

            if boxes:
                processed_target = {
                    "boxes": torch.tensor(boxes, dtype=torch.float32).to(device),
                    "labels": torch.tensor(labels, dtype=torch.int64).to(device),
                }
                processed_targets.append(processed_target)
                valid_images.append(images[i])

        if not processed_targets:
            continue

        images = valid_images

        # Forward + backward
        loss_dict = model(images, processed_targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        # Logging
        total_loss += losses.item()
        count += 1

        if writer:
            for k, v in loss_dict.items():
                writer.add_scalar(f"Loss/{k}", v.item(), epoch * len(data_loader) + step)
            writer.add_scalar("Loss/total", losses.item(), epoch * len(data_loader) + step)

    avg_loss = total_loss / count if count > 0 else 0.0
    print(f"[Epoch {epoch}] Média da loss: {avg_loss:.4f}")

    # Logging por época
    if writer:
        writer.add_scalar("EpochLoss/total", avg_loss, epoch)

In [ ]:
# Training loop
# SummaryWriter for logging
writer = SummaryWriter("runs/fasterrcnn_exp")

num_epochs = 5
for epoch in range(num_epochs):
    train_one_epoch(model, optimizer, train_loader, device, epoch, writer) 
    lr_scheduler.step()

    # Save model
    model_path = f"fasterrcnn_resnet50_epoch_{epoch + 1}.pth"
    torch.save(model.state_dict(), model_path)
    print(f"Model saved: {model_path}")

writer.close()

## Versão Corrigida para Treino

In [ ]:
import torch
import torchvision
from torch.utils.data import DataLoader
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.datasets import CocoDetection
import torchvision.transforms.functional as F
from pytorch_lightning import LightningModule, Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pycocotools.cocoeval import COCOeval
from pycocotools.coco import COCO
import os
import json
import tempfile

# Dataset wrapper
class CocoTransform:
    def __call__(self, image, target):
        image = F.to_tensor(image)
        return image, target

class CustomCocoDetection(CocoDetection):
    def __init__(self, root, annFile):
        super().__init__(root, annFile, transforms=CocoTransform())

# LightningModule
class FasterRCNNLitModule(LightningModule):
    def __init__(self, num_classes):
        super().__init__()
        self.save_hyperparameters()
        self.model = self.get_model(num_classes)
        self.val_ann_file = "images/val/annotations.json"
        self.outputs_buffer = []

    def get_model(self, num_classes):
        weights = torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT
        model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=weights)
        in_features = model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
        return model

    def forward(self, images, targets=None):
        return self.model(images, targets)

    def training_step(self, batch, batch_idx):
        images, targets = self.preprocess_batch(batch)
        loss_dict = self.model(images, targets)
        total_loss = sum(loss for loss in loss_dict.values())
        self.log("train_loss", total_loss, on_step=False, on_epoch=True)
        return total_loss

    def validation_step(self, batch, batch_idx):
        images, targets = self.preprocess_batch(batch)
        outputs = self.model(images)

        coco_preds = []
        for img_id, output in enumerate(outputs):
            boxes = output['boxes'].cpu().tolist()
            scores = output['scores'].cpu().tolist()
            labels = output['labels'].cpu().tolist()
            image_id = targets[img_id]["image_id"].item() if "image_id" in targets[img_id] else img_id
            for box, score, label in zip(boxes, scores, labels):
                x_min, y_min, x_max, y_max = box
                coco_preds.append({
                    "image_id": image_id,
                    "category_id": label,
                    "bbox": [x_min, y_min, x_max - x_min, y_max - y_min],
                    "score": score
                })

        self.outputs_buffer.extend(coco_preds)

    def on_validation_epoch_end(self):
        if not self.outputs_buffer:
            return

        with tempfile.NamedTemporaryFile(mode='w+', delete=False, suffix='.json') as f:
            json.dump(self.outputs_buffer, f)
            pred_file = f.name

        coco_gt = COCO(self.val_ann_file)
        coco_dt = coco_gt.loadRes(pred_file)

        # Verificar se image_ids batem certo
        pred_image_ids = set(p['image_id'] for p in self.outputs_buffer)
        gt_image_ids = set(coco_gt.getImgIds())
        assert pred_image_ids.issubset(gt_image_ids), "Predicted image_ids don't match ground truth."

        coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()

        map_50 = coco_eval.stats[1]  # AP@IoU=0.50
        self.log("val_mAP", map_50, prog_bar=True)

        self.outputs_buffer.clear()

    def preprocess_batch(self, batch):
        images, targets = batch
        processed_images = []
        processed_targets = []
        for i in range(len(images)):
            boxes = []
            labels = []
            for obj in targets[i]:
                bbox = obj['bbox']
                x, y, w, h = bbox
                if w > 0 and h > 0:
                    boxes.append([x, y, x + w, y + h])
                    labels.append(obj['category_id'])
            if boxes:
                image_id = obj.get("image_id", i) if isinstance(obj, dict) else i
                processed_images.append(images[i].to(self.device))
                processed_targets.append({
                    "boxes": torch.tensor(boxes, dtype=torch.float32).to(self.device),
                    "labels": torch.tensor(labels, dtype=torch.int64).to(self.device),
                    "image_id": torch.tensor(image_id)
                })
        return processed_images, processed_targets

    def configure_optimizers(self):
        params = [p for p in self.model.parameters() if p.requires_grad]
        optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
        return [optimizer], [scheduler]

# Dataset paths
train_dataset = CustomCocoDetection("images/train", "images/train/annotations.json")
val_dataset = CustomCocoDetection("images/val", "images/val/annotations.json")

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

# Callbacks
early_stop_callback = EarlyStopping(monitor="val_mAP", mode="max", patience=5, verbose=True)
checkpoint_callback = ModelCheckpoint(dirpath="checkpoints", filename="best_model", save_top_k=1, monitor="val_mAP", mode="max")

# Trainer
model = FasterRCNNLitModule(num_classes=2)  # 1 classe + background
trainer = Trainer(
    max_epochs=20,
    accelerator="auto",
    callbacks=[early_stop_callback, checkpoint_callback],
    log_every_n_steps=10
)

trainer.fit(model, train_loader, val_loader)


loading annotations into memory...
Done (t=0.19s)
creating index...
index created!
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name  | Type       | Params | Mode 
---------------------------------------------
0 | model | FasterRCNN | 41.3 M | train
---------------------------------------------
41.1 M    Trainable params
222 K     Non-trainable params
41.3 M    Total params
165.197   Total estimated model params size (MB)
189       Modules in train mode
0         Modules in eval mode


Sanity Checking DataLoader 0: 100%|██████████| 2/2 [00:26<00:00,  0.08it/s]loading annotations into memory...
Done (t=0.04s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.03s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.14s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 1

c:\Users\Dell\OneDrive - Confidencial Imobiliario\Documentos\ComputerVision\Task 3\1st_part\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.


Epoch 0:   0%|          | 0/361 [00:00<?, ?it/s] 

c:\Users\Dell\OneDrive - Confidencial Imobiliario\Documentos\ComputerVision\Task 3\1st_part\.venv\Lib\site-packages\pytorch_lightning\utilities\data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 3. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


Epoch 0:   1%|          | 4/361 [02:43<4:02:32,  0.02it/s, v_num=1]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [22]:
!pip install pytorch-lightning

  Using cached attrs-25.3.0-py3-none-any.whl.metadata (10 kB)
   ---------------------------------------- 0.0/823.1 kB ? eta -:--:--
   ---------------------------------------- 823.1/823.1 kB 9.9 MB/s eta 0:00:00
Using cached attrs-25.3.0-py3-none-any.whl (63 kB)
   ---------------------------------------- 0.0/962.5 kB ? eta -:--:--
   --------------------------------------- 962.5/962.5 kB 14.5 MB/s eta 0:00:00

   --- ------------------------------------  1/11 [multidict]
   --- ------------------------------------  1/11 [multidict]
   ------- --------------------------------  2/11 [lightning-utilities]
   ------- --------------------------------  2/11 [lightning-utilities]
   ------- --------------------------------  2/11 [lightning-utilities]
   ------- --------------------------------  2/11 [lightning-utilities]
   -------------- -------------------------  4/11 [attrs]
   -------------- -------------------------  4/11 [attrs]
   -------------- -------------------------  4/11 [attrs